In [8]:
import pandas as pd
import requests

for year in range(2022, 2025):
    url = f"https://api.jolpi.ca/ergast/f1/{year}/races.json"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    races = data["MRData"]["RaceTable"]["Races"]
    df = pd.json_normalize(races)

    reduced_df = df[[
        "season",
        "round",
        "raceName",
        "Circuit.circuitName",
        "Circuit.Location.locality",
        "Circuit.Location.country",
        "date",
        "time",
    ]]

    reduced_df.to_json(
        path_or_buf=f"../data/raw/races/{year}_races.json",
        orient="records",
    )

In [ ]:
import pandas as pd
import requests

for year in range(2022, 2025):
    races = pd.read_json(f"../data/raw/races/{year}_races.json")
    rounds = len(races["round"].unique())
    
    print(f"{year} :total rounds: {rounds}")

    df_final = pd.DataFrame()

    for race in range(1, rounds + 1):
        url = f"https://api.jolpi.ca/ergast/f1/{year}/{race}/results.json"
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        races = data["MRData"]["RaceTable"]["Races"]

        df = pd.json_normalize(
            races,
            record_path="Results",
            meta=["season", "round"],
        )

        reduced_df = df[[
            "season",
            "round",
            "position",
            "points",
            "grid",
            "status",
            "Driver.driverId",
            "Constructor.constructorId",
        ]]

        df_final = pd.concat([df_final, reduced_df])

    df_final.to_json(
        path_or_buf=f"../data/raw/results/{year}_results.json",
        orient="records",
    )

2022 :total rounds: 22
2023 :total rounds: 22
2024 :total rounds: 24


HTTPError: 429 Client Error: Too Many Requests for url: https://api.jolpi.ca/ergast/f1/2024/21/results.json

In [23]:
import pandas as pd

for year in [2022, 2023, 2024]:
    races = pd.read_json(f"../data/raw/races/{year}_races.json")
    results = pd.read_json(f"../data/raw/results/{year}_results.json")

    df = results.merge(races, on=["season", "round"], how="left")

    df["position"] = pd.to_numeric(df["position"], errors="coerce")
    df["grid"] = pd.to_numeric(df["grid"], errors="coerce")
    df["round"] = pd.to_numeric(df["round"])
    df["race_datetime"] = pd.to_datetime(df["date"].astype(str) + " " + df["time"])

    df = df.sort_values(["Driver.driverId", "season", "round"])
    df["driver_last_race_position"] = (
        df.groupby("Driver.driverId")["position"]
        .shift(1)
        .astype("Int64")
    )
    df["driver_avg_position_last_3_races"] = (
        df.groupby("Driver.driverId")["position"]
        .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    )

    df = df.dropna(subset=["position", "grid"])
    df = df.drop(columns=["date", "time"])

    # e.g. "assume they finish where they start" when no history
    df["driver_last_race_position"] = df["driver_last_race_position"].fillna(df["grid"])
    df["driver_avg_position_last_3_races"] = df["driver_avg_position_last_3_races"].fillna(df["grid"])

    df = df.rename(columns={
        "Driver.driverId": "driver_id",
        "Constructor.constructorId": "constructor_id",
        "Circuit.circuitName": "circuit",
        "Circuit.Location.locality": "locality",
        "Circuit.Location.country": "country",
    })

    df.to_json(
        path_or_buf=f"../data/processed/{year}_season.json",
        orient="records",
        date_format="iso",
    )

    print(f"{year}: {len(df)} rows")

2022: 440 rows
2023: 440 rows
2024: 479 rows


In [25]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

df = pd.DataFrame()
for year in [2022, 2023, 2024]:
    df_temp = pd.read_json(f"../data/processed/{year}_season.json")
    df = pd.concat([df, df_temp])
    
# train = df[df["round"] < 18]
# test = df[df["round"] >= 18]

train = df[df["season"] < 2024]
test = df[df["season"] >= 2024]

training_features= train[[
    "grid", 
    "driver_last_race_position", 
    "driver_avg_position_last_3_races", 
    "driver_id", 
    "constructor_id", 
    "circuit", 
    "locality", 
    "country", 
    "race_datetime"]]
training_target = train["position"]

test_features = test[[
    "grid", 
    "driver_last_race_position", 
    "driver_avg_position_last_3_races", 
    "driver_id", 
    "constructor_id", 
    "circuit", 
    "locality", 
    "country", 
    "race_datetime"]]
test_target = test["position"]

# Baseline finishing position = grid position
baseline_model = mean_absolute_error(test_target, test_features["grid"])
print(f"Baseline MAE (predict position = grid): {baseline_model:.2f}")


preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["driver_id", "constructor_id", "circuit", "locality", "country", "race_datetime"]),
    ("num", "passthrough", ["grid", "driver_last_race_position", "driver_avg_position_last_3_races"]),
])

model = Pipeline([
    ("prep", preprocessor),
    ("reg", RandomForestRegressor(n_estimators=400, random_state=42)),
])

model.fit(training_features, training_target)
preds = model.predict(test_features)

print(f"Model MAE: {mean_absolute_error(test_target, preds):.2f}")


Baseline MAE (predict position = grid): 2.91
Model MAE: 3.00
